# Code mẫu — Feature engineering

Một feature chỉ dùng được nếu nó tính được từ những thông tin đã có vào lúc cần dự đoán. Các ví dụ dưới đây kiểm điều đó với dữ liệu thời gian và dữ liệu danh mục. Ví dụ cuối so hai bộ feature trên cùng các fold, để thấy một feature tốt giúp model nhiều tới đâu.

## Ví dụ 1: Để 23 giờ nằm cạnh 0 giờ

Giờ trong ngày quay vòng sau 24 giờ. Nếu để số giờ thô, 23 giờ và 0 giờ cách nhau tới 23 đơn vị, dù thực tế chúng chỉ cách nhau một giờ. Ta đặt mỗi giờ lên một vòng tròn bằng hai tọa độ sin và cos, rồi đo lại khoảng cách.

In [ ]:
import numpy as np

hours = np.array([0., 1., 12., 23.])
angle = 2 * np.pi * hours / 24
encoded = np.column_stack([np.sin(angle), np.cos(angle)])
print("(giờ, sin, cos):", np.column_stack([hours, encoded]).round(3).tolist())
print("khoảng cách 23–0 thô / chu kỳ:",
      abs(hours[3] - hours[0]), round(float(np.linalg.norm(encoded[3] - encoded[0])), 3))
print("khoảng cách 12–0 chu kỳ:", round(float(np.linalg.norm(encoded[2] - encoded[0])), 3))
print("norm mỗi hàng:", np.linalg.norm(encoded, axis=1).round(3).tolist())
assert np.allclose(np.linalg.norm(encoded, axis=1), 1.)

(giờ, sin, cos): [[0.0, 0.0, 1.0], [1.0, 0.259, 0.966], [12.0, 0.0, -1.0], [23.0, -0.259, 0.966]]
khoảng cách 23–0 thô / chu kỳ: 23.0 0.261
khoảng cách 12–0 chu kỳ: 2.0
norm mỗi hàng: [1.0, 1.0, 1.0, 1.0]


Trên vòng tròn, 23 giờ chỉ cách 0 giờ khoảng 0.261, bằng đúng khoảng cách từ 0 giờ tới 1 giờ. Còn 12 giờ nằm ở phía đối diện, cách 0 giờ đúng 2, xa nhất có thể. Cần đủ cả hai cột. Nếu chỉ dùng sin, 0 giờ và 12 giờ đều cho sin bằng 0, và ta không phân biệt được nửa đêm với giữa trưa. Con số 24 là hiểu biết sẵn có về dữ liệu, nên phép biến đổi này không cần học gì từ train.

## Ví dụ 2: Feature quá khứ không được đổi khi sửa target hiện tại

Hai cửa hàng A và B có số đơn theo từng giờ, nhưng các hàng trong file bị xáo trộn thứ tự. Để dự đoán số đơn ở một giờ, ta dùng hai feature: số đơn của giờ ngay trước, gọi là lag, và trung bình của hai giờ trước đó. Muốn tính đúng, phải sắp các hàng theo cửa hàng rồi theo giờ trước đã.

In [ ]:
import numpy as np
import pandas as pd

history = pd.DataFrame({
    "shop": ["A", "B", "A", "A", "B", "A"],
    "hour": [3, 2, 1, 4, 1, 2],
    "sales": [6., 9., 4., 10., 2., 8.],
})

def historical_features(frame):
    result = frame.sort_values(["shop", "hour"]).reset_index(drop=True).copy()
    result["lag_1"] = result.groupby("shop")["sales"].shift(1)
    result["mean_prev_2"] = result.groupby("shop")["sales"].transform(
        lambda series: series.shift(1).rolling(2, min_periods=2).mean()
    )
    return result

features = historical_features(history)
print(features.to_string(index=False))
at_hour_3 = (features["shop"] == "A") & (features["hour"] == 3)
print("A giờ 3: lag / mean:",
      features.loc[at_hour_3, ["lag_1", "mean_prev_2"]].to_numpy().tolist())
assert features.loc[at_hour_3, "mean_prev_2"].iloc[0] == (4 + 8) / 2

shop  hour  sales  lag_1  mean_prev_2
   A     1    4.0    NaN          NaN
   A     2    8.0    4.0          NaN
   A     3    6.0    8.0          6.0
   A     4   10.0    6.0          7.0
   B     1    2.0    NaN          NaN
   B     2    9.0    2.0          NaN
A giờ 3: lag / mean: [[8.0, 6.0]]


Ở cửa hàng A giờ 3, lag là 8, tức số đơn giờ 2. Trung bình là 6, tính từ giờ 1 và giờ 2 là 4 và 8. Con số 6 trong cột `sales` của giờ 3 chưa hề được dùng. Những giờ đầu của mỗi cửa hàng chưa có đủ lịch sử, nên NaN ở đó là đúng. Đừng lấy số liệu của giờ sau để lấp NaN cho đẹp.

Có một cách kiểm rò rỉ rất dễ hiểu: sửa target của A giờ 3 từ 6 thành 600, rồi tính lại. Feature dùng để dự đoán giờ 3 phải giữ nguyên, vì lúc dự đoán ta chưa biết target ấy. Feature của giờ 4 thì được phép đổi, vì khi dự đoán từng giờ một, tới giờ 4 ta đã thấy số đơn thật của giờ 3.

In [ ]:
changed = history.copy()
changed.loc[(changed["shop"] == "A") & (changed["hour"] == 3), "sales"] = 600.
changed_features = historical_features(changed)
valid_columns = ["lag_1", "mean_prev_2"]
print("feature giờ 3 không đổi:", np.allclose(
    features.loc[at_hour_3, valid_columns], changed_features.loc[at_hour_3, valid_columns]
))
wrong = features.groupby("shop")["sales"].transform(lambda s: s.rolling(2).mean())
wrong_changed = changed_features.groupby("shop")["sales"].transform(lambda s: s.rolling(2).mean())
print("rolling chưa shift tại giờ 3:",
      float(wrong[at_hour_3].iloc[0]), "->", float(wrong_changed[at_hour_3].iloc[0]))
assert np.allclose(features.loc[at_hour_3, valid_columns],
                   changed_features.loc[at_hour_3, valid_columns])

feature giờ 3 không đổi: True
rolling chưa shift tại giờ 3: 7.0 -> 304.0


Feature đúng không đổi. Còn bản rolling quên `shift` thì nhảy từ 7 lên 304, chỉ vì ta sửa đúng con số đang cần đoán. Đó là bằng chứng nó đã chứa target bên trong.

Lưu ý thêm một trường hợp. Nếu đề bắt dự đoán cùng lúc cả giờ 3 lẫn giờ 4, thì lúc đó ta chưa biết số đơn giờ 3, nên không được dùng nó làm lag cho giờ 4. Khi ấy phải dựng lại feature theo đúng quy trình dự đoán mới, không dùng nguyên bảng ở trên.

## Ví dụ 3: Mã hiếm có thể giấu chính đáp án trong feature

Có sáu mã khách hàng, mỗi mã chỉ xuất hiện một lần, và target là số đơn. Target encoding thay mỗi mã bằng trung bình target của mã đó. Nếu học bảng mã từ toàn bộ train rồi áp lại lên chính train, mà không làm mượt, thì feature của mỗi hàng chính là target của hàng đó. Ta so cách ấy với cross-fitting ba fold. Trong cross-fitting, mỗi hàng chỉ được mã hóa bằng nhãn của các fold khác.

In [ ]:
import numpy as np
from sklearn.preprocessing import TargetEncoder

X = np.array([["A"], ["B"], ["C"], ["D"], ["E"], ["F"]], dtype=object)
y = np.array([5., 8., 11., 20., 25., 30.])
unsafe_encoder = TargetEncoder(target_type="continuous", smooth=0., cv=3, shuffle=False)
unsafe_train = unsafe_encoder.fit(X, y).transform(X)
safe_encoder = TargetEncoder(target_type="continuous", smooth=0., cv=3, shuffle=False)
cross_fitted = safe_encoder.fit_transform(X, y)
new_customer = safe_encoder.transform(np.array([["NEW"]], dtype=object))
print("target:", y.tolist())
print("fit rồi transform train:", unsafe_train.ravel().round(3).tolist())
print("fit_transform cross-fitting:", cross_fitted.ravel().round(3).tolist())
print("mã mới / mean train:", round(float(new_customer[0, 0]), 3), round(float(y.mean()), 3))
assert np.allclose(unsafe_train.ravel(), y)
assert np.allclose(cross_fitted[:2].ravel(), y[2:].mean())

target: [5.0, 8.0, 11.0, 20.0, 25.0, 30.0]
fit rồi transform train: [5.0, 8.0, 11.0, 20.0, 25.0, 30.0]
fit_transform cross-fitting: [21.5, 21.5, 17.0, 17.0, 11.0, 11.0]
mã mới / mean train: 16.5 16.5


Dòng thứ hai là đáp án chép lại nguyên văn. Model học trên feature này sẽ đạt điểm train tuyệt đối, rồi sụp đổ khi gặp khách mới. Với cross-fitting, hai hàng đầu nhận 21.5, là trung bình nhãn của bốn khách C, D, E, F, chứ không phải 5 và 8 của chính chúng. Mã chưa từng gặp nhận trung bình của toàn bộ train, 16.5.

Chú ý là trong scikit-learn, `fit(X, y).transform(X)` và `fit_transform(X, y)` cho kết quả khác nhau với TargetEncoder. Chỉ `fit_transform` mới làm cross-fitting. Cách chia fold ở đây giả định các hàng độc lập với nhau. Nếu dữ liệu có nhóm hoặc có thứ tự thời gian, phải chia theo nhóm hoặc theo thời gian. Khi chạy CV, encoder cũng phải nằm trong Pipeline, để phần validation của mỗi fold không góp vào bảng mã.

## Ví dụ 4: Ước tính thời gian di chuyển bằng quãng đường chia tốc độ

Đầu vào gồm quãng đường tính bằng km, tốc độ tính bằng km/h, và loại tuyến. Target là số phút di chuyển. Quãng đường và tốc độ đều biết trước chuyến đi, nên feature `60 × km / tốc độ` là hợp lệ. Ta so một model chỉ dùng các cột gốc với một model có thêm feature này, trên cùng ba fold.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(31)
distance = rng.uniform(2, 30, 90)
speed = rng.uniform(20, 60, 90)
route = np.where(np.arange(90) % 2 == 0, "city", "outer")
y = 60 * distance / speed + 4 * (route == "city") + rng.normal(0, 0.5, 90)
frame = pd.DataFrame({"km": distance, "speed": speed, "route": route})
frame["travel_minutes"] = 60 * frame["km"] / frame["speed"]
folds = list(KFold(3, shuffle=True, random_state=4).split(frame))
for name, numeric_columns in [
    ("cột gốc", ["km", "speed"]),
    ("thêm tỷ số", ["km", "speed", "travel_minutes"]),
]:
    preprocessing = ColumnTransformer([
        ("number", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_columns),
        ("route", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["route"]),
    ])
    pipeline = make_pipeline(preprocessing, Ridge(alpha=1.))
    mae = -cross_val_score(pipeline, frame, y, cv=folds,
                           scoring="neg_mean_absolute_error", n_jobs=1)
    print(name, "MAE từng fold:", mae.round(3).tolist(), "mean:", round(float(mae.mean()), 3))

cột gốc MAE từng fold: [3.824, 4.734, 3.706] mean: 4.088
thêm tỷ số MAE từng fold: [0.538, 0.74, 0.606] mean: 0.628


Thêm một cột tỷ số, MAE trung bình giảm từ 4.088 xuống 0.628 phút, và fold nào cũng giảm. Lý do là model tuyến tính chỉ biết cộng các cột với nhau, nên tự nó không học được phép chia. Ta đưa sẵn phép chia vào thành một cột, và model chỉ việc dùng.

Imputer, scaler và bảng one-hot đều được học lại trên phần train của mỗi fold. Feature tỷ số chỉ dùng thông tin có trước chuyến đi. Nếu ai đó tạo feature từ thời gian thực tế của chính chuyến đi, code vẫn chạy, CV có thể rất đẹp, nhưng đó là dùng target để đoán target.

So xong, ta fit Pipeline có tỷ số trên toàn bộ dữ liệu rồi dự đoán cho một chuyến mới. Chuyến này đi tuyến `rural`, loại tuyến chưa từng xuất hiện trong train. Sau khi biến đổi, số cột phải giữ nguyên như lúc học.

In [ ]:
fitted = pipeline.fit(frame, y)  # Biến pipeline lúc này là cấu hình có tỷ số, từ vòng lặp cuối.
new_trip = pd.DataFrame({"km": [10.], "speed": [40.], "route": ["rural"]})
new_trip["travel_minutes"] = 60 * new_trip["km"] / new_trip["speed"]
transformed = fitted[0].transform(new_trip)
print("tên cột:", fitted[0].get_feature_names_out().tolist())
print("shape / mã tuyến mới:", transformed.shape, transformed[0, -2:].tolist())
print("dự đoán phút:", fitted.predict(new_trip).round(3).tolist())
assert np.allclose(transformed[0, -2:], 0.)

tên cột: ['number__km', 'number__speed', 'number__travel_minutes', 'route__route_city', 'route__route_outer']
shape / mã tuyến mới: (1, 5) [0.0, 0.0]
dự đoán phút: [17.281]


Nhờ `handle_unknown="ignore"`, tuyến rural nhận 0 ở cả hai cột `city` và `outer`, và shape vẫn là `(1, 5)`. Chuyến 10 km với tốc độ 40 km/h mất 15 phút đi đường. Model đoán 17.281 phút. Phần chênh khoảng 2.3 phút nằm giữa mức 0 phút của tuyến outer và mức 4 phút của tuyến city, vì với cả hai cột bằng 0, model không có thông tin riêng nào về tuyến rural. Không có nhãn thật của tuyến rural thì ta chưa biết dự đoán này sai bao nhiêu. Còn một chuyện nên định trước: nếu tốc độ có thể bằng 0, phải quyết định cách xử lý trước khi chia, thay vì để giá trị vô cùng lọt vào Pipeline.